# OSC Pick-and-Place Training (Colab) — Hierarchical

Phase 3, built on `5-arm_project_osc`'s 100%-reliable Phase 2 (Grasp) model. Two end-to-end training attempts (plain BC pretraining, then ongoing BC regularization) both failed to reliably learn the pick sub-skill jointly with the longer place sequence — see `IMP_NOTES.md` incidents #3-#4. This notebook now uses a **hierarchical** approach instead: the pick portion (reach → grasp → lift → hold) is handled entirely by the FROZEN, already-proven 5-arm_project_osc model (`pretrained/sac_franka_grasp_frozen.zip`, committed to this repo) via `HierarchicalPickPlaceEnv` — nothing is learned there anymore, it's just run. Only the genuinely new sub-task (carry, release, settle) is trained, starting from a state where the object is already reliably held. See `envs/hierarchical_pick_place_env.py` and incident #5.

Cell 5.5 collects place-only demonstrations (`collect_place_demonstrations.py` — the pick portion inside it is already auto-piloted, so this only scripts the carry/release/settle sequence). Cell 6 runs `train_hierarchical_place_bc_parallel.py`.

**Runtime type**: a plain CPU runtime is fine — no need to select a GPU.

**Why Drive is mounted**: Colab's local disk is wiped whenever the runtime disconnects or recycles — checkpoints are symlinked into Google Drive below so they survive a disconnect; only re-run cells 1-4 to resume watching a run, and cells 5.5-6 again to re-collect demonstrations and continue/restart training.

In [1]:
# Cell 1 — mount Google Drive (checkpoints and the final model save both
# land here, not on Colab's ephemeral local disk)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Cell 2 — clone the repo. Leave the token prompt blank if the repo is public;
# paste a GitHub Personal Access Token (repo scope) if it's private.
import getpass, os

REPO_URL = "https://github.com/kaustubhadhe1206/Arm-OSC-Pick-and-Place.git"
REPO_DIR = "Arm-OSC-Pick-and-Place"

token = getpass.getpass("GitHub token (leave blank if repo is public): ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {clone_url}

%cd {REPO_DIR}

Cloning into 'Arm-OSC-Pick-and-Place'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 100 (delta 10), reused 95 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 4.86 MiB | 10.26 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Arm-OSC-Pick-and-Place


In [3]:
# Cell 3 — point checkpoints at Drive via a symlink, so the training script's
# existing `save_path="./checkpoints/"` transparently writes to Drive instead
# of Colab's local (ephemeral) disk, with no changes needed to the script
# itself.
import os

DRIVE_CKPT_DIR = "/content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

if os.path.islink("checkpoints") or os.path.isdir("checkpoints"):
    !rm -rf checkpoints
!ln -s {DRIVE_CKPT_DIR} checkpoints

print("Checkpoints will be saved to:", DRIVE_CKPT_DIR)

Checkpoints will be saved to: /content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints


In [4]:
# Cell 4 — install dependencies. torch is already preinstalled on Colab (CUDA
# build) — that's fine, the training script forces device="cpu" regardless.
!pip install -q mujoco gymnasium stable-baselines3

In [5]:
# Cell 5 — sanity check: how many CPU cores does this runtime actually have?
# (5-arm_project_osc's Colab sessions showed 2 vCPUs on the free tier.)
import os
print("CPU count:", os.cpu_count())

CPU count: 2


In [ ]:
# Cell 5.5 — collect PLACE-ONLY demonstrations. The pick portion is
# auto-piloted internally by the frozen model (see
# envs/hierarchical_pick_place_env.py), so this script only scripts the
# carry/descend/release/settle sequence, starting from an
# already-successfully-grasped state. Locally this reached 100% success
# (5/5) — much more reliable than the full-sequence scripted routine's
# 75-85%, since pick failures are now retried invisibly instead of
# counting against this script's success rate at all. Only needs to run
# once per Colab session; place_demonstrations.npz persists in this
# session's local disk for the rest of it.
!python collect_place_demonstrations.py 300

In [ ]:
# Cell 6 — run hierarchical place-only training. This streams SB3's
# logging table live and blocks until 1,000,000 steps complete or the
# runtime disconnects — checkpoints every 12,500 steps land in Drive via
# the symlink either way. This sub-task is much shorter/simpler than the
# full sequence (no pick to learn at all), so watch ep_len_mean/
# ep_rew_mean and consider stopping early if it plateaus at a good success
# rate well before 1M steps.
!python train_hierarchical_place_bc_parallel.py

In [ ]:
# Cell 7 — only relevant if cell 6 finished without disconnecting: the FINAL
# model.save() writes to the repo directory (Colab's local disk), not
# checkpoints/ — copy it to Drive too so it isn't lost.
!cp sac_franka_hierarchical_place_bc_parallel.zip /content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints/ 2>/dev/null || echo "Not found yet — training may not have completed."